In [ ]:
# ==========================================
# NOTEBOOK 1B: FL SEGMENTATION
# Methods: FedProx (20 rounds)
# Requires: isic2018-seg-part-a as input
# ==========================================

!pip install -q kagglehub scikit-learn tqdm segmentation-models-pytorch

import os, copy, json, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image
from collections import OrderedDict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. HARDCODED PATHS
# ==========================================
PART_A    = "/kaggle/input/datasets/peesarisathvikreddy/isic2018-seg-part-a"
ISIC_PATH = "/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation"

assert os.path.exists(os.path.join(PART_A, "meta.json")), \
    f"meta.json not found at {PART_A}"
print(f"✅ Part A : {PART_A}")
print(f"✅ ISIC   : {ISIC_PATH}")

TRAIN_IMG_DIR  = os.path.join(
    ISIC_PATH, "ISIC2018_Task1-2_Training_Input")
TRAIN_MASK_DIR = os.path.join(
    ISIC_PATH, "ISIC2018_Task1_Training_GroundTruth")

print(f"Train images : {len(os.listdir(TRAIN_IMG_DIR))} files")
print(f"Train masks  : {len(os.listdir(TRAIN_MASK_DIR))} files")

# ==========================================
# 2. RELOAD META + DATA FROM PART A
# ==========================================
with open(os.path.join(PART_A, "meta.json")) as f:
    meta = json.load(f)

majority_classes_per_client = meta['majority_classes_per_client']
IMG_SIZE                    = meta['img_size']
MAIN_FRAC                   = meta['main_frac']

train_df   = pd.read_csv(os.path.join(PART_A, "train_df.csv"))
test_df    = pd.read_csv(os.path.join(PART_A, "test_df.csv"))
client_dfs = [pd.read_csv(os.path.join(PART_A, f"client_{i}_train.csv"))
              for i in range(3)]
client_sizes = [len(c) for c in client_dfs]

print(f"\nTrain : {len(train_df)} | Test : {len(test_df)}")
print(f"Client sizes : {client_sizes}")

# ==========================================
# 3. TRANSFORMS + DATASET
# ==========================================
IMG_SIZE = meta['img_size']

img_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
mask_tf = transforms.Compose([
    transforms.Resize(
        (IMG_SIZE, IMG_SIZE),
        interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

class SegDataset(Dataset):
    def __init__(self, df, img_tf=None, mask_tf=None, noise=0.0):
        self.df=df.reset_index(drop=True)
        self.img_tf=img_tf; self.mask_tf=mask_tf; self.noise=noise
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=Image.open(row['image']).convert("RGB")
        mask=Image.open(row['mask']).convert("L")
        if self.img_tf:  img=self.img_tf(img)
        if self.mask_tf: mask=self.mask_tf(mask)
        mask=(mask>0.5).float()
        if self.noise>0:
            img+=torch.randn_like(img)*self.noise/255.0
            img=torch.clamp(img,0,1)
        return img, mask, row['size_label']

def make_loader(df, noise=0.0, batch_size=8, shuffle=False):
    return DataLoader(
        SegDataset(df, img_tf, mask_tf, noise),
        batch_size=batch_size, shuffle=shuffle,
        num_workers=2, pin_memory=True)

NOISE       = [0.0, 15.0, 10.0]
dataloaders = [make_loader(cdf, noise=NOISE[i])
               for i, cdf in enumerate(client_dfs)]
val_df      = train_df.sample(frac=0.2, random_state=42)
val_loader  = make_loader(val_df)
test_loader = make_loader(test_df)
print(f"\nVal:{len(val_df)} | Test:{len(test_df)}")

# ==========================================
# 4. MODEL — U-Net (ResNet-18 encoder)
# ==========================================
try:
    import segmentation_models_pytorch as smp
    def get_model():
        return smp.Unet(
            encoder_name="resnet18", encoder_weights="imagenet",
            in_channels=3, classes=1).to(device)
    print("\nUsing SMP U-Net")
except ImportError:
    class DoubleConv(nn.Module):
        def __init__(self,ic,oc):
            super().__init__()
            self.c=nn.Sequential(
                nn.Conv2d(ic,oc,3,padding=1,bias=False),
                nn.BatchNorm2d(oc),nn.ReLU(inplace=True),
                nn.Conv2d(oc,oc,3,padding=1,bias=False),
                nn.BatchNorm2d(oc),nn.ReLU(inplace=True))
        def forward(self,x): return self.c(x)
    class UNet(nn.Module):
        def __init__(self,features=[64,128,256,512]):
            super().__init__()
            self.downs=nn.ModuleList(); self.ups=nn.ModuleList()
            self.pool=nn.MaxPool2d(2); c=3
            for f in features: self.downs.append(DoubleConv(c,f)); c=f
            self.bottleneck=DoubleConv(features[-1],features[-1]*2)
            for f in reversed(features):
                self.ups.append(nn.ConvTranspose2d(f*2,f,2,stride=2))
                self.ups.append(DoubleConv(f*2,f))
            self.final=nn.Conv2d(features[0],1,1)
        def forward(self,x):
            skips=[]
            for d in self.downs: x=d(x); skips.append(x); x=self.pool(x)
            x=self.bottleneck(x); skips=skips[::-1]
            for i in range(0,len(self.ups),2):
                x=self.ups[i](x); sk=skips[i//2]
                if x.shape!=sk.shape: x=F.interpolate(x,size=sk.shape[2:])
                x=self.ups[i+1](torch.cat([sk,x],dim=1))
            return self.final(x)
    def get_model(): return UNet().to(device)
    print("\nUsing custom U-Net")

_m=get_model(); _x=torch.randn(2,3,IMG_SIZE,IMG_SIZE).to(device)
print(f"Output shape: {_m(_x).shape}"); del _m,_x
torch.cuda.empty_cache()

# ==========================================
# 5. LOSS + METRICS
# ==========================================
class DiceBCELoss(nn.Module):
    def __init__(self,smooth=1e-6):
        super().__init__()
        self.bce=nn.BCEWithLogitsLoss(); self.s=smooth
    def forward(self,logits,targets):
        bce=self.bce(logits,targets); p=torch.sigmoid(logits)
        i=(p*targets).sum(dim=(2,3))
        d=1-(2*i+self.s)/(p.sum(dim=(2,3))+targets.sum(dim=(2,3))+self.s)
        return bce+d.mean()

criterion = DiceBCELoss()

def compute_metrics(logits, masks, thr=0.5):
    preds=(torch.sigmoid(logits)>thr).float(); s=1e-6
    tp=(preds*masks).sum(dim=(2,3))
    fp=(preds*(1-masks)).sum(dim=(2,3))
    fn=((1-preds)*masks).sum(dim=(2,3))
    tn=((1-preds)*(1-masks)).sum(dim=(2,3))
    return {
        'dice':        ((2*tp+s)/(2*tp+fp+fn+s)).mean().item(),
        'iou':         ((tp+s)/(tp+fp+fn+s)).mean().item(),
        'pixel_acc':   ((tp+tn+s)/(tp+fp+fn+tn+s)).mean().item(),
        'sensitivity': ((tp+s)/(tp+fn+s)).mean().item(),
        'specificity': ((tn+s)/(tn+fp+s)).mean().item(),
    }

def evaluate(model, loader):
    model.eval(); all_m=[]
    with torch.no_grad():
        for batch in loader:
            imgs,masks=batch[0].to(device),batch[1].to(device)
            all_m.append(compute_metrics(model(imgs),masks))
    return {k:float(np.mean([m[k] for m in all_m])) for k in all_m[0]}

# ==========================================
# 6. LOCAL TRAINING + AGGREGATION
# ==========================================
MU = 0.001

def train_local(model, loader, gw,
                local_epochs=3, lr=1e-4,
                mu=0.0):
    model.train()
    opt=optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-5)
    for _ in range(local_epochs):
        for batch in loader:
            imgs,masks=batch[0].to(device),batch[1].to(device)
            opt.zero_grad(); loss=criterion(model(imgs),masks)
            if mu>0:
                loss+=(mu/2)*sum(
                    torch.sum((p-gw[n].to(device))**2)
                    for n,p in model.named_parameters()
                    if n in gw and p.dtype.is_floating_point)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()
    return {k:v.cpu().clone() for k,v in model.state_dict().items()}

def fedavg_agg(lw):
    avg=OrderedDict()
    for k in lw[0]:
        t=[w[k] for w in lw]
        avg[k]=torch.stack(t).mean(0) \
               if t[0].dtype.is_floating_point else t[0].clone()
    return avg

def fedprox_agg(lw, gw, mu=MU):
    avg=fedavg_agg(lw)
    for k in avg:
        if avg[k].dtype.is_floating_point:
            avg[k]=avg[k]-mu*(avg[k]-gw[k].cpu())
    return avg

# ==========================================
# 7. GENERIC FL RUNNER
# ==========================================
def run_fl(name, dataloaders, client_sizes,
           rounds=20, local_epochs=3, lr=1e-4,
           save_clients=True):

    gm  = get_model()
    gw  = {k:v.cpu() for k,v in gm.state_dict().items()}
    cms = [get_model() for _ in range(3)]
    best_dice, best_gw = -1.0, None
    history = []

    for rnd in range(rounds):

        # FedProx
        lw=[]
        for cid,loader in enumerate(dataloaders):
            cms[cid].load_state_dict(gw,strict=True)
            lw.append(train_local(cms[cid],loader,gw,
                                  local_epochs=local_epochs,
                                  lr=lr, mu=MU))
        gw=fedprox_agg(lw,gw)
        gm.load_state_dict(gw,strict=True)

        val_m=evaluate(gm,val_loader)
        history.append({
            'round':rnd, 'aggregation':name,
            'dice':val_m['dice'], 'iou':val_m['iou'],
            'pixel_acc':val_m['pixel_acc'],
            'sensitivity':val_m['sensitivity'],
            'specificity':val_m['specificity'],
        })
        print(f"  Round {rnd+1}/{rounds} {name} → "
              f"Dice:{val_m['dice']:.4f}  IoU:{val_m['iou']:.4f}")

        if val_m['dice']>best_dice:
            best_dice=val_m['dice']; best_gw=copy.deepcopy(gw)

    gm.load_state_dict(best_gw,strict=True)

    if save_clients:
        for cid in range(3):
            sm=get_model()
            sm.load_state_dict(cms[cid].state_dict(), strict=True)
            torch.save(sm.state_dict(),
                       f"/kaggle/working/client_{name}_{cid}.pth")

    fm=evaluate(gm,test_loader)
    return history, gm, cms, fm

# ==========================================
# 8. MAIN RUN — FedProx
# ==========================================
ROUNDS       = 20
LOCAL_EPOCHS = 3
LR           = 1e-4
METHODS      = ['fedprox']

print(f"\n{'='*60}")
print(f"MAIN TRAINING — frac={MAIN_FRAC}, rounds={ROUNDS}")
print(f"{'='*60}")

all_history   = []
final_models  = {}
final_metrics = {}

for name in METHODS:
    print(f"\n{'='*50}\nRUNNING {name.upper()}\n{'='*50}")
    hist, gm, cms, fm = run_fl(
        name, dataloaders, client_sizes,
        rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, lr=LR,
        save_clients=True)

    for rec in hist: all_history.append(rec)
    final_models[name]  = gm
    final_metrics[name] = fm

    torch.save(gm.state_dict(),
               f"/kaggle/working/final_model_{name}.pth")
    print(f"✅ {name} → Dice={fm['dice']:.4f}  IoU={fm['iou']:.4f}")

# ==========================================
# 9. SAVE + COPY PART A FILES FORWARD
# ==========================================
import shutil

# Copy all Part A files needed by Notebook 2
files_to_copy = [
    "test_df.csv",
    "train_df.csv",
    "meta.json",
    "fl_results_part_a.csv",
    "client_0_train.csv",
    "client_1_train.csv",
    "client_2_train.csv",
    "client_0_test_split.csv",
    "client_1_test_split.csv",
    "client_2_test_split.csv",
    "noniid_distribution.png",
]
print("\nCopying Part A files forward...")
for fname in files_to_copy:
    src = os.path.join(PART_A, fname)
    dst = f"/kaggle/working/{fname}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"  ✅ {fname}")
    else:
        print(f"  ⚠️  {fname} not found in Part A")

# Save Part B history
pd.DataFrame(all_history).to_csv(
    "/kaggle/working/fl_results_part_b.csv", index=False)

# Convergence plot
df_h=pd.DataFrame(all_history)
plt.figure(figsize=(10,6))
colors={'fedprox':'#2ca02c'}
for name in METHODS:
    sub=df_h[df_h['aggregation']==name]
    plt.plot(sub['round']+1, sub['dice'],
             marker='o', color=colors[name], label=name.upper())
plt.xlabel('Round'); plt.ylabel('Dice Score (Validation)')
plt.title('FL Segmentation — FedProx Convergence')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig("/kaggle/working/convergence_part_b.png", dpi=150)
plt.show()

print("\n✅ Notebook 1B complete.")
print("\nFiles saved:")
print("  fl_results_part_b.csv")
print("  final_model_fedprox.pth")
print("  client_fedprox_0.pth")
print("  client_fedprox_1.pth")
print("  client_fedprox_2.pth")
print("  convergence_part_b.png")
print("  (all Part A files copied forward)")
print("\n➡ Save output as Kaggle dataset: 'isic2018-seg-part-b'")
print("➡ Add both part-a and part-b as inputs to Notebook 2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/154.8 kB ? eta -:--:--


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.7 MB/s eta 0:00:00


Using device: cuda
✅ Part A : /kaggle/input/datasets/peesarisathvikreddy/isic2018-seg-part-a
✅ ISIC   : /kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation
Train images : 2596 files
Train masks  : 2596 files



Train : 2075 | Test : 519
Client sizes : [688, 687, 700]

Val:415 | Test:519



Using SMP U-Net


...

  Round 4/20 ditto → Dice:0.8945  IoU:0.8262


  Round 5/20 ditto → Dice:0.8952  IoU:0.8255


  Round 6/20 ditto → Dice:0.8938  IoU:0.8260


  Round 7/20 ditto → Dice:0.9008  IoU:0.8353


  Round 8/20 ditto → Dice:0.8999  IoU:0.8350
